# 05 · Determinism & Time

**Focus:** eliminating flaky tests caused by the clock, using `time-machine`.

Code that calls `datetime.now()` is a classic source of **flaky tests**: a test that passes today
may fail next month, because "now" keeps moving. You can't assert on behavior that depends on the
current time unless you can *control* the current time.

`time-machine` does exactly that. `time_machine.travel(...)` freezes `datetime.now()` (and friends)
to an instant you choose, for the duration of a `with` block or a decorated test. Your code keeps
calling `datetime.now()` normally — it just gets the frozen value.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### Time-dependent code
Whether a subscription has expired depends on *now*.

In [2]:
%%writefile nb05_subscription.py
from datetime import datetime

def is_expired(expiry: datetime) -> bool:
    '''True if the subscription's expiry date is in the past.'''
    return datetime.now() >= expiry

Writing nb05_subscription.py


### The problem, then the fix
The first test is **naive** — it only works because we picked dates far from today, and it would
break in edge cases. The real tests **freeze time** with `time_machine.travel`, making "now" a
fixed, known instant. Now we can assert precisely around the boundary.

In [3]:
%%writefile test_nb05.py
from datetime import datetime
import time_machine
from nb05_subscription import is_expired

# --- Freezing time with a context manager ---
def test_not_yet_expired():
    with time_machine.travel("2025-06-15 12:00:00"):
        # "now" is pinned to 2025-06-15. Expiry is later -> not expired.
        assert is_expired(datetime(2025, 12, 31)) is False

def test_already_expired():
    with time_machine.travel("2025-06-15 12:00:00"):
        # Expiry is in the past relative to frozen now -> expired.
        assert is_expired(datetime(2025, 1, 1)) is True

# --- Freezing time with a decorator (same effect, cleaner for whole tests) ---
@time_machine.travel("2030-01-01 00:00:00")
def test_boundary_exact_moment():
    # now == expiry exactly. is_expired uses >=, so this is True.
    assert is_expired(datetime(2030, 1, 1, 0, 0, 0)) is True

Writing test_nb05.py


### Run it — the same result every day, forever

In [4]:
!pytest -v test_nb05.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 3 items                                                              

test_nb05.py::test_not_yet_expired PASSED                                [ 33%]
test_nb05.py::test_already_expired PASSED                                [ 66%]
test_nb05.py::test_boundary_exact_moment PASSED                          [100%]

============================== 3 passed in 0.03s =======

### ⚠️ Common pitfall — freezing isn't the same as advancing
`time_machine.travel("2025-06-15")` *pins* "now" to a single instant; it doesn't tick forward on its
own. If your test needs time to *pass* (a token expiring, a cache aging), grab the traveller with
`as traveller:` and call `traveller.shift(...)` to move the clock deliberately.

### 🔬 Try it yourself
This test freezes time *before* the expiry date, then jumps 10 days forward. **Predict:** what does
`is_expired` return before and after the `shift`? Run it — then try changing the shift to `days=3`
(before expiry) and re-run.

In [5]:
%%writefile test_nb05_tryit.py
from datetime import datetime, timedelta
import time_machine
from nb05_subscription import is_expired

def test_time_advances():
    expiry = datetime(2025, 6, 20)
    with time_machine.travel("2025-06-15") as traveller:
        assert is_expired(expiry) is False        # now = Jun 15 -> not expired
        traveller.shift(timedelta(days=10))       # jump the clock to Jun 25
        assert is_expired(expiry) is True         # now = Jun 25 -> expired!

Writing test_nb05_tryit.py


In [6]:
!pytest -v test_nb05_tryit.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item                                                               

test_nb05_tryit.py::test_time_advances PASSED                            [100%]

============================== 1 passed in 0.03s ===============================


### What you learned
- Any code calling `datetime.now()` is untestable *until you control the clock*.
- `with time_machine.travel("YYYY-MM-DD ..."):` freezes now inside the block.
- `@time_machine.travel(...)` does the same for an entire test function.
- Frozen time lets you test exact boundaries (`now == expiry`) deterministically — no more flakes.

That wraps the first half of the course. Next: **exceptions and logs**.